# Đánh giá WDMamba trên SOTS-ITS và SOTS-OTS

Notebook sử dụng cách tải trọng số và triển khai trong `Bench.ipynb`.

- Tải SOTS Indoor và Outdoor bằng KaggleHub.
- Tải `WDMamba_ckpts.zip` từ Google Drive.
- Dùng trọng số RESIDE-6K: `reside6k_32.15.pth`.
- Chạy suy luận với **batch size 16**.
- Đo thời gian DWT, LFRN, IWT, DEN và tổng thời gian mô hình.
- Tính PSNR, SSIM và ước lượng GFLOPs.

Mặc định thay đổi kích thước ảnh về **256 × 256** để phân tích độ phức tạp. FPS chỉ tính thời gian mô hình, không gồm đọc ảnh từ đĩa. Điểm số ở kích thước này không phải điểm SOTS ở độ phân giải gốc.


In [ ]:

from pathlib import Path
import os, sys, time, json, zipfile, subprocess, warnings
import torch

def find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "Bench.ipynb").is_file() and (p / "solution" / "WDMamba").is_dir():
            return p.resolve()
    p = Path(r"E:/repoE/VDT_GD2")
    if (p / "solution" / "WDMamba").is_dir():
        return p.resolve()
    raise FileNotFoundError("Không tìm thấy thư mục dự án VDT_GD2")

ROOT = find_root()
WDMAMBA_ROOT = ROOT / "solution" / "WDMamba"
OUTPUT_ROOT = ROOT / "outputs" / "wdmamba_sots_benchmark"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 16
IMAGE_SIZE = 256
MAX_IMAGES = 0
WARMUP_BATCHES = 3
NUM_WORKERS = 0 if os.name == "nt" else min(4, os.cpu_count() or 1)
PIN_MEMORY = DEVICE.startswith("cuda")

print(ROOT, DEVICE, BATCH_SIZE, IMAGE_SIZE)


## Cài các thư viện phục vụ đánh giá

Cell này cài các thư viện còn thiếu. Môi trường chạy cần có PyTorch, CUDA, `mamba-ssm` và các phụ thuộc kiến trúc tương thích như môi trường WDMamba trong `Bench.ipynb`.


In [ ]:

packages = ["kagglehub", "gdown", "thop", "pandas", "scikit-image", "Pillow"]
for pkg in packages:
    module = {"Pillow":"PIL", "scikit-image":"skimage"}.get(pkg, pkg)
    try:
        __import__(module)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)


## Tải SOTS-ITS và SOTS-OTS

Tải bộ dữ liệu SOTS từ KaggleHub, sau đó tìm các thư mục `indoor` và `outdoor` chứa ảnh `hazy` và `clear`. SOTS-ITS là tập kiểm thử trong nhà; SOTS-OTS là tập kiểm thử ngoài trời. Cell này không tải các tập huấn luyện ITS/OTS.


In [ ]:

import kagglehub, cv2, numpy as np
SOTS_KAGGLE_DATASET = "balraj98/synthetic-objective-testing-set-sots-reside"
sots_download_root = Path(kagglehub.dataset_download(SOTS_KAGGLE_DATASET))
print("SOTS:", sots_download_root)

def image_files(folder):
    return sorted(p for p in Path(folder).iterdir()
                  if p.is_file() and p.suffix.lower() in {".png",".jpg",".jpeg",".bmp",".tif",".tiff"})

def find_domain(root, domain):
    for p in [root / domain, *[q for q in root.rglob(domain) if q.is_dir()]]:
        if (p / "clear").is_dir() and (p / "hazy").is_dir():
            return p
    raise FileNotFoundError(f"Không tìm thấy {domain}/clear và {domain}/hazy")

SOTS_DOMAINS = {
    "SOTS-ITS": find_domain(sots_download_root, "indoor"),
    "SOTS-OTS": find_domain(sots_download_root, "outdoor"),
}
for label, root in SOTS_DOMAINS.items():
    print(label, root, len(image_files(root/"hazy")), "ảnh có haze")


## Tải và giải nén trọng số WDMamba

Dùng cùng mã tệp Google Drive và trọng số `reside6k_32.15.pth` như trong `Bench.ipynb`. Tái sử dụng tệp ZIP và trọng số đã giải nén nếu có sẵn.


In [ ]:

sys.path.insert(0, str(ROOT / "utils"))
from wdmamba_checkpoints import download_drive

DRIVE_FILE_ID = "1Vm-1mtMSc9YIUICq4nWKu-4AHCtYnPpF"
CKPT_DIR = ROOT / "checkpoints" / "wdmamba"
CKPT_ZIP = CKPT_DIR / "WDMamba_ckpts.zip"
UNZIP_DIR = CKPT_DIR / "unzipped"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

if not (CKPT_ZIP.is_file() and zipfile.is_zipfile(CKPT_ZIP)):
    download_drive(DRIVE_FILE_ID, CKPT_ZIP)
else:
    print("Tái sử dụng:", CKPT_ZIP)

checkpoint_path = next(UNZIP_DIR.rglob("reside6k_32.15.pth"), None) if UNZIP_DIR.exists() else None
if checkpoint_path is None:
    UNZIP_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(CKPT_ZIP) as archive:
        archive.extractall(UNZIP_DIR)
    checkpoint_path = next(UNZIP_DIR.rglob("reside6k_32.15.pth"), None)
if checkpoint_path is None:
    raise FileNotFoundError("Không tìm thấy trọng số reside6k_32.15.pth")
print("Trọng số:", checkpoint_path)


## Tạo bộ nạp dữ liệu và tải mô hình

Ghép ảnh haze với ảnh sạch theo phần tên trước dấu gạch dưới. Bộ nạp dữ liệu sử dụng batch size 16; batch cuối có thể chứa ít hơn 16 ảnh. Mặc định đặt số worker bằng 0 trên Windows để tránh lỗi multiprocessing trong Jupyter.


In [ ]:

from torch.utils.data import Dataset, DataLoader
from evaluate_wdmamba import load_model

def collect_pairs(root, label):
    clear = {p.stem:p for p in image_files(root/"clear")}
    pairs = []
    for hazy in image_files(root/"hazy"):
        gt = clear.get(hazy.stem.split("_")[0])
        if gt is None:
            raise FileNotFoundError(f"Không tìm thấy ảnh sạch tương ứng với {hazy.name}")
        pairs.append((label, hazy.stem, hazy, gt))
    return pairs[:MAX_IMAGES] if MAX_IMAGES > 0 else pairs

PAIRSETS = {label:collect_pairs(root,label) for label,root in SOTS_DOMAINS.items()}

def read_rgb(path):
    image = cv2.imread(str(path), cv2.IMREAD_COLOR)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    if IMAGE_SIZE:
        image = cv2.resize(image, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_AREA)
    return image

class SOTSDataset(Dataset):
    def __init__(self, pairs): self.pairs = pairs
    def __len__(self): return len(self.pairs)
    def __getitem__(self, i):
        label, name, hazy_path, gt_path = self.pairs[i]
        hazy = torch.from_numpy(read_rgb(hazy_path).copy()).float().permute(2,0,1)/255
        gt = torch.from_numpy(read_rgb(gt_path).copy()).float().permute(2,0,1)/255
        return hazy, gt, label, name

def loader(label):
    return DataLoader(SOTSDataset(PAIRSETS[label]), batch_size=BATCH_SIZE,
                      shuffle=False, num_workers=NUM_WORKERS,
                      pin_memory=PIN_MEMORY, persistent_workers=NUM_WORKERS>0)

model = load_model(checkpoint_path, DEVICE).eval()
print("Đã tải mô hình; số block trong mỗi nhóm DEN:", model.de_blocks)


## Chạy suy luận và đo thời gian từng giai đoạn

Tái hiện luồng xử lý của WDMamba: DWT → LFRN → IWT → DEN. IWT ghép LL đã phục hồi với ba thành phần HL, LH, HH lấy trực tiếp từ ảnh haze đầu vào.

Trên GPU, thời gian được đo bằng CUDA events và có đồng bộ. Tổng thời gian trong bảng là tổng thời gian của bốn giai đoạn đã đo riêng.


In [ ]:

from basicsr.archs.wavelet import DWT, IWT
dwt, iwt = DWT().to(DEVICE), IWT().to(DEVICE)

def sync():
    if DEVICE.startswith("cuda"): torch.cuda.synchronize()

def timed(fn):
    sync()
    if DEVICE.startswith("cuda"):
        start, end = torch.cuda.Event(True), torch.cuda.Event(True)
        start.record(); value = fn(); end.record(); sync()
        return value, start.elapsed_time(end)/1000
    start = time.perf_counter(); value = fn()
    return value, time.perf_counter()-start

def forward_stages(wrapper, x):
    net = wrapper.restoration_network
    n = x.shape[0]
    x_dwt, t_dwt = timed(lambda:dwt(x))
    x_ll, x_high = x_dwt[:n], x_dwt[n:]

    def run_lfrn():
        blocks = []
        x0 = net.down_group0(net.layer0(x_ll)); blocks.append(x0)
        x1 = net.down_group1(net.layer1(x0)); blocks.append(x1)
        x2 = net.down_group2(net.layer2(x1)); blocks.append(x2)
        x3 = net.down_group3(net.layer3(x2)); blocks.append(x3)
        x4 = net.down_group4(net.layer4(x3))
        y3 = net.up_group4(net.layer_3(x4,blocks[-1]))
        y2 = net.up_group3(net.layer_2(y3,blocks[-2]))
        y1 = net.up_group2(net.layer_1(y2,blocks[-3]))
        y0 = net.up_group1(net.layer_0(y1,blocks[-4]))
        return net.last(y0) + x_ll

    output_ll, t_lfrn = timed(run_lfrn)
    stage1, t_iwt = timed(lambda:iwt(torch.cat((output_ll,x_high),dim=0)))
    final, t_den = timed(lambda:net.DE(stage1))
    return final, {"DWT":t_dwt,"LFRN":t_lfrn,"IWT":t_iwt,"DEN":t_den,
                   "Model total":t_dwt+t_lfrn+t_iwt+t_den}

sample = next(iter(loader("SOTS-ITS")))[0].to(DEVICE, non_blocking=PIN_MEMORY)
with torch.inference_mode():
    test_out, test_time = forward_stages(model, sample)
print(sample.shape, test_out.shape, test_time)


## Đánh giá thời gian, FPS, PSNR và SSIM

Chạy khởi động trước khi ghi số đo. Thời gian được quy đổi thành mili giây trên mỗi ảnh; FPS là tổng số ảnh chia cho tổng thời gian các giai đoạn mô hình. Đây là thông lượng ở batch 16, không phải độ trễ xử lý một ảnh ở batch 1.


In [ ]:

from skimage.metrics import peak_signal_noise_ratio, structural_similarity
import pandas as pd

def image_metrics(out, gt):
    a = out.detach().float().clamp(0,1).permute(0,2,3,1).cpu().numpy()
    b = gt.detach().float().clamp(0,1).permute(0,2,3,1).cpu().numpy()
    psnr = [peak_signal_noise_ratio(y,x,data_range=1.0) for x,y in zip(a,b)]
    ssim = [structural_similarity(y,x,channel_axis=-1,data_range=1.0) for x,y in zip(a,b)]
    return psnr, ssim

def benchmark(label):
    dl = loader(label)
    for _ in range(WARMUP_BATCHES):
        try:
            batch = next(iter(dl))[0].to(DEVICE, non_blocking=PIN_MEMORY)
        except StopIteration:
            break
        with torch.inference_mode():
            _ = forward_stages(model, batch)
    totals = {k:0.0 for k in ["DWT","LFRN","IWT","DEN","Model total"]}
    psnr, ssim, count = [], [], 0
    for hazy, gt, _, _ in dl:
        hazy, gt = hazy.to(DEVICE, non_blocking=PIN_MEMORY), gt.to(DEVICE, non_blocking=PIN_MEMORY)
        with torch.inference_mode():
            out, t = forward_stages(model, hazy)
        for k in totals: totals[k] += t[k]
        p, s = image_metrics(out, gt)
        psnr += p; ssim += s; count += hazy.shape[0]
    row = {"dataset":label,"images":count,"batch_size":BATCH_SIZE,"image_size":IMAGE_SIZE,
           "PSNR":float(np.mean(psnr)),"SSIM":float(np.mean(ssim))}
    row.update({k+"_ms_per_image":1000*v/count for k,v in totals.items()})
    row["Model_FPS"] = count/totals["Model total"]
    return row

timing_df = pd.DataFrame([benchmark("SOTS-ITS"), benchmark("SOTS-OTS")])
display(timing_df)
timing_df.to_csv(OUTPUT_ROOT/"sots_timing_metrics.csv", index=False)


## Ước lượng GFLOPs theo từng giai đoạn

`thop` đếm các phép tính mà thư viện hỗ trợ, chủ yếu convolution và linear. Bảng dùng quy ước **FLOPs = 2 × MACs** cho phần đếm bằng `thop`, và một ước lượng riêng cho phép biến đổi Haar.

Các toán tử tùy chỉnh như selective scan, FFT và phép biến đổi wavelet bên trong mạng có thể bị bỏ sót. Vì vậy các giá trị này là ước lượng chưa đầy đủ, không nên coi là toàn bộ FLOPs chính xác hoặc so sánh trực tiếp với paper nếu khác quy ước đếm. Cột GFLOPs cho batch 16 bằng GFLOPs mỗi ảnh nhân với 16.


In [ ]:

from thop import profile
from torch import nn

class LFRNOnly(nn.Module):
    def __init__(self,r): super().__init__(); self.r=r
    def forward(self,ll):
        n=self.r; b=[]
        x0=n.down_group0(n.layer0(ll)); b.append(x0)
        x1=n.down_group1(n.layer1(x0)); b.append(x1)
        x2=n.down_group2(n.layer2(x1)); b.append(x2)
        x3=n.down_group3(n.layer3(x2)); b.append(x3)
        x4=n.down_group4(n.layer4(x3))
        x3=n.up_group4(n.layer_3(x4,b[-1]))
        x2=n.up_group3(n.layer_2(x3,b[-2]))
        x1=n.up_group2(n.layer_1(x2,b[-3]))
        x0=n.up_group1(n.layer_0(x1,b[-4]))
        return n.last(x0)+ll

class DENOnly(nn.Module):
    def __init__(self,r): super().__init__(); self.de=r.DE
    def forward(self,x): return self.de(x)

def gflops(m,x):
    try:
        macs, params = profile(m.eval(), inputs=(x,), verbose=False)
        return 2*macs/1e9, params/1e6, "thop"
    except Exception as e:
        print("Cảnh báo khi đếm FLOPs bằng thop:", type(e).__name__, e)
        return float("nan"), float("nan"), "failed"

dummy = torch.zeros(1,3,IMAGE_SIZE,IMAGE_SIZE,device=DEVICE)
dummy_ll = torch.zeros(1,3,IMAGE_SIZE//2,IMAGE_SIZE//2,device=DEVICE)
lfrn_g,lfrn_p,lfrn_src = gflops(LFRNOnly(model.restoration_network).to(DEVICE),dummy_ll)
den_g,den_p,den_src = gflops(DENOnly(model.restoration_network).to(DEVICE),dummy)
haar_g = 2*(5*3*IMAGE_SIZE*IMAGE_SIZE)/1e9

flops_df = pd.DataFrame([
 {"stage":"DWT","GFLOPs_per_image":haar_g,"params_M":0,"source":"analytic Haar"},
 {"stage":"LFRN","GFLOPs_per_image":lfrn_g,"params_M":lfrn_p,"source":lfrn_src},
 {"stage":"IWT","GFLOPs_per_image":haar_g,"params_M":0,"source":"analytic Haar"},
 {"stage":"DEN","GFLOPs_per_image":den_g,"params_M":den_p,"source":den_src},
])
flops_df["GFLOPs_per_batch16"] = flops_df["GFLOPs_per_image"]*BATCH_SIZE
display(flops_df)

flops_summary = {
 "device":DEVICE, "checkpoint":str(checkpoint_path), "batch_size":BATCH_SIZE,
 "image_size":IMAGE_SIZE,
 "estimated_full_GFLOPs_per_image":float(flops_df["GFLOPs_per_image"].sum()),
 "estimated_full_GFLOPs_per_batch16":float(flops_df["GFLOPs_per_batch16"].sum()),
 "note":"Selective scan và FFT có thể chưa được thop đếm đầy đủ; dùng thời gian thực đo và FPS làm chỉ số triển khai chính.",
}
print(json.dumps(flops_summary,indent=2,ensure_ascii=False))
flops_df.to_csv(OUTPUT_ROOT/"sots_stage_gflops.csv",index=False)
(OUTPUT_ROOT/"benchmark_summary.json").write_text(
 json.dumps({"timing":timing_df.to_dict(orient="records"),"flops":flops_summary},
            indent=2,ensure_ascii=False), encoding="utf-8")


## Diễn giải kết quả

- Cả SOTS-ITS và SOTS-OTS dùng trọng số RESIDE-6K theo cách đánh giá chuyển miền trong `Bench.ipynb`; đây không phải kết quả kiểm thử RESIDE-6K của paper.
- PSNR/SSIM được tính trên ảnh đã thay đổi kích thước theo `IMAGE_SIZE`.
- Thời gian không bao gồm tải dữ liệu từ mạng, đọc ảnh từ đĩa hoặc tính các chỉ số chất lượng.
- GFLOPs chỉ là ước lượng; selective scan và FFT có thể chưa được đếm đầy đủ. Ưu tiên thời gian thực đo khi so sánh triển khai.
- Kết quả được lưu dưới `outputs/wdmamba_sots_benchmark/` gồm CSV thời gian, CSV GFLOPs và JSON tổng hợp.
